In [ ]:
# =============================================================================
# Monthly stack export — 12 tasks total
# Each output image has bands named: temp_max_20240101, rh_min_20240101, ...
# Spatial context fully preserved — same ERA5 grid as daily approach.
# =============================================================================

import ee
import time

# ── 0. Auth ───────────────────────────────────────────────────────────────────
KEY_FILE        = 'ru-thesis-2026-66238f2c1197.json'
SERVICE_ACCOUNT = 'ru-thesis-gee-api@ru-thesis-2026.iam.gserviceaccount.com'
PROJECT         = 'ru-thesis-2026'

credentials = ee.ServiceAccountCredentials(SERVICE_ACCOUNT, KEY_FILE)
ee.Initialize(credentials, project=PROJECT)

# ── 1. Master Grid ────────────────────────────────────────────────────────────
era5_sample = (ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
               .first())
GRID_PROJ   = era5_sample.select(0).projection()
GRID_SCALE  = GRID_PROJ.nominalScale()
print(f"Master Grid initialized at {GRID_SCALE.getInfo():.0f} m scale.")

# ── 2. Study area ─────────────────────────────────────────────────────────────
ontario_fc   = (ee.FeatureCollection("FAO/GAUL/2015/level1")
                .filter(ee.Filter.eq('ADM0_NAME', 'Canada'))
                .filter(ee.Filter.eq('ADM1_NAME', 'Ontario')))
ontario_geom = ontario_fc.geometry().simplify(4500)

# reproject kept — snapping vector province boundary to ERA5 grid 
PROVINCE_RASTER = (ontario_fc
                   .map(lambda f: f.set('province_code', 9))
                   .reduceToImage(['province_code'], ee.Reducer.first())
                   .reproject(crs=GRID_PROJ)
                   .rename('province_id')
                   .toByte())

# reproject kept — defining mask on ERA5 grid 
era5_coverage = (ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
                 .filterDate('2024-07-01', '2024-07-02')
                 .first()
                 .select('temperature_2m')
                 .mask()
                 .reproject(crs=GRID_PROJ)
                 .rename('era5_coverage'))

ONTARIO_MASK    = (PROVINCE_RASTER.eq(9).And(era5_coverage).selfMask())
PROVINCE_RASTER = PROVINCE_RASTER.updateMask(ONTARIO_MASK)
canada_geom     = ontario_geom

# ── 3. Date sequences ─────────────────────────────────────────────────────────
EXPORT_YEAR  = 2010
VALID_YEARS  = list(range(EXPORT_YEAR - 1, EXPORT_YEAR + 1))

STUDY_START  = ee.Date(f'{EXPORT_YEAR}-01-01')
STUDY_END    = ee.Date(f'{EXPORT_YEAR}-12-31')
BUFFER_START = ee.Date(f'{EXPORT_YEAR - 1}-12-02')
BUFFER_END   = STUDY_END.advance(1, 'day')

N_BUFFER_DAYS = BUFFER_END.difference(BUFFER_START, 'day')
buffer_dates  = ee.List.sequence(0, N_BUFFER_DAYS.subtract(1)) \
                        .map(lambda n: BUFFER_START.advance(n, 'day'))

N_STUDY_DAYS = STUDY_END.difference(STUDY_START, 'day').add(1)
study_dates  = ee.List.sequence(0, N_STUDY_DAYS.subtract(1)) \
                       .map(lambda n: STUDY_START.advance(n, 'day'))

# ── 4. Fire points — system:time_start set so filterDate works correctly ──────
quality_filter = ee.Filter.And(
    ee.Filter.inList('YEAR', VALID_YEARS),
    ee.Filter.neq('MONTH',    0),
    ee.Filter.neq('DAY',      0),
    ee.Filter.neq('LATITUDE', 0),
    ee.Filter.neq('LONGITUDE',0))


# ── 4. Fire points 
def encode_fire_point(f):
    date       = ee.Date.fromYMD(f.get('YEAR'), f.get('MONTH'), f.get('DAY'))
    date_num   = ee.Number.parse(date.format('YYYYMMdd'))
    cause_str  = ee.String(f.get('CAUSE')).trim()
    cause_code = ee.Number(
        ee.Algorithms.If(cause_str.equals('U'),    1,
        ee.Algorithms.If(cause_str.equals('H'),    2,
        ee.Algorithms.If(cause_str.equals('N'),    3,
        ee.Algorithms.If(cause_str.equals('H-PB'), 4,
        ee.Algorithms.If(cause_str.equals('RE'),   5, 0))))))
    return f.set({
        'START_DATE_num'     : date_num,
        'CAUSE_code'         : cause_code,
        'date_millis'        : date.millis(),
        'system:time_start'  : date.millis()   
    })

fire_points = (ee.FeatureCollection(
                   "projects/ru-thesis-2026/assets/Fire_point_20250519")
               .filter(quality_filter)
               .filterBounds(ontario_geom)
               .map(encode_fire_point))

Master Grid initialized at 11132 m scale.


In [ ]:


from itertools import groupby

EXPORT_FOLDER_MONTHLY = f"projects/{PROJECT}/assets/Ontario_Monthly_2010-2024"
RECOVERY_DAYS         = 180
BATCH_SIZE            = 50
SLEEP_SEC             = 10

try:
    ee.data.createAsset({'type': 'ImageCollection'}, EXPORT_FOLDER_MONTHLY)
    print(f"Created: {EXPORT_FOLDER_MONTHLY}")
except ee.EEException:
    print(f"Already exists: {EXPORT_FOLDER_MONTHLY}")


def build_one_day(d_str):
    """
    Build all bands for a single day, with date suffix on every band name.
    Returns an ee.Image with bands named e.g. temp_max_20240715.

    Parameters
    ----------
    d_str : str  e.g. '20240715'
    """
    d        = ee.Date.parse('YYYYMMdd', d_str)
    date_num = ee.Number.parse(d_str)
    suffix   = f'_{d_str}'

    # ── ERA5 daily ────────────────────────────────────────────────────────────
    era5 = (ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
            .filterDate(d, d.advance(1, 'day'))
            .filterBounds(ontario_geom)
            .first())

    temp_max  = era5.select('temperature_2m_max').subtract(273.15).rename(f'temp_max{suffix}')
    temp_mean = era5.select('temperature_2m')    .subtract(273.15).rename(f'temp_mean{suffix}')
    td_mean   = era5.select('dewpoint_temperature_2m')    .subtract(273.15)
    td_min    = era5.select('dewpoint_temperature_2m_min').subtract(273.15)
    t_max_c   = era5.select('temperature_2m_max')         .subtract(273.15)

    A, B    = 17.625, 243.04
    rh_mean = (td_mean.multiply(A).divide(td_mean.add(B)).exp()
               .divide(temp_mean.multiply(A).divide(temp_mean.add(B)).exp())
               .multiply(100).rename(f'rh_mean{suffix}'))
    rh_min  = (td_min.multiply(A).divide(td_min.add(B)).exp()
               .divide(t_max_c.multiply(A).divide(t_max_c.add(B)).exp())
               .multiply(100).rename(f'rh_min{suffix}'))

    A2, B2, SCALE = 17.27, 237.3, 0.6108
    vpd = (temp_mean.rename('t').multiply(A2).divide(temp_mean.rename('t').add(B2)).exp().multiply(SCALE)
           .subtract(td_mean.multiply(A2).divide(td_mean.add(B2)).exp().multiply(SCALE))
           .rename(f'vpd_mean{suffix}'))

    u10   = era5.select('u_component_of_wind_10m')
    v10   = era5.select('v_component_of_wind_10m')
    wspd  = u10.pow(2).add(v10.pow(2)).sqrt().rename(f'wind_speed_mean{suffix}')
    precip = era5.select('total_precipitation_sum').multiply(1000).rename(f'precip_sum{suffix}')
    solar  = era5.select('surface_net_solar_radiation_sum').divide(1e6).rename(f'solar_rad_mj{suffix}')

    # ── Lags — filter the full daily_weather_col which covers buffer period ───
    # daily_weather_col is defined below as an ImageCollection of daily images
    # built from ERA5 DAILY_AGGR over the buffer + study period
    rh_temp_7d = (daily_weather_col
                  .filterDate(d.advance(-6, 'day'), d.advance(1, 'day'))
                  .select(['rh_mean', 'temp_mean'])
                  .mean()
                  .rename([f'rh_7d{suffix}', f'temp_7d{suffix}']))

    precip_30d = (daily_weather_col
                  .filterDate(d.advance(-29, 'day'), d.advance(1, 'day'))
                  .select('precip_sum')
                  .sum()
                  .rename(f'precip_30d{suffix}'))

    # ── Recovery ──────────────────────────────────────────────────────────────
    recent_fires  = fire_points.filterDate(d.advance(-RECOVERY_DAYS, 'day'), d)
    latest_millis = (recent_fires
                     .reduceToImage(['date_millis'], ee.Reducer.max())
                     .rename('last_burn_millis'))
    days_since    = (ee.Image(d.millis())
                     .subtract(latest_millis)
                     .divide(1000 * 60 * 60 * 24))
    recovery_val  = (days_since.divide(RECOVERY_DAYS)
                     .pow(0.5).clamp(0, 1)
                     .rename(f'recovery_value{suffix}')
                     .unmask(1))

    # ── Ignition targets ──────────────────────────────────────────────────────
    daily_ignitions = fire_points.filter(
                          ee.Filter.equals('START_DATE_num', date_num))
    target_ignition = (daily_ignitions
                       .reduceToImage(['START_DATE_num'], ee.Reducer.count())
                       .reproject(crs=GRID_PROJ, scale=GRID_SCALE)
                       .gt(0).unmask(0).byte()
                       .rename(f'target_ignition{suffix}'))
    fire_cause = (daily_ignitions
                  .reduceToImage(['CAUSE_code'], ee.Reducer.max())
                  .reproject(crs=GRID_PROJ, scale=GRID_SCALE)
                  .unmask(0).byte()
                  .rename(f'fire_cause{suffix}'))

    return (temp_max
            .addBands([rh_min, rh_mean, vpd, wspd, precip, solar,
                       rh_temp_7d, precip_30d, recovery_val,
                       target_ignition, fire_cause])
            .updateMask(ONTARIO_MASK))


# ── Build the lightweight daily collection for lag computation ────────────────
# This covers buffer + study period so lags are correct from Jan 1.
# Each image has rh_mean, temp_mean, precip_sum for lag inputs only.

def build_lag_input(current_date):
    """Lightweight daily image used only as lag input — not exported."""
    d    = ee.Date(current_date)
    era5 = (ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
            .filterDate(d, d.advance(1, 'day'))
            .filterBounds(ontario_geom)
            .first())

    temp_mean = era5.select('temperature_2m').subtract(273.15).rename('temp_mean')
    td_mean   = era5.select('dewpoint_temperature_2m').subtract(273.15)
    t_mean_c  = era5.select('temperature_2m').subtract(273.15)

    A, B    = 17.625, 243.04
    rh_mean = (td_mean.multiply(A).divide(td_mean.add(B)).exp()
               .divide(t_mean_c.multiply(A).divide(t_mean_c.add(B)).exp())
               .multiply(100).rename('rh_mean'))

    precip = era5.select('total_precipitation_sum').multiply(1000).rename('precip_sum')

    return (temp_mean.addBands([rh_mean, precip])
            .updateMask(ONTARIO_MASK)
            .set('system:time_start', d.millis()))


daily_weather_col = ee.ImageCollection(buffer_dates.map(build_lag_input))
print("Lag input collection defined.")


# ── Monthly export loop ───────────────────────────────────────────────────────
study_date_strings = (study_dates
                      .map(lambda d: ee.Date(d).format('YYYYMMdd'))
                      .getInfo())

def get_month(d_str):
    return d_str[:6]   # '20240715' → '202407'

print(f"Submitting monthly export tasks …")

for month_str, dates in groupby(study_date_strings, key=get_month):
    dates     = list(dates)
    n_bands   = len(dates) * 11   # 11 output bands per day

    # Build all daily images for this month and stack into one image
    daily_images = [build_one_day(d_str) for d_str in dates]

    # Stack all days — first image as base, rest added as bands
    monthly_stack = daily_images[0]
    for img in daily_images[1:]:
        monthly_stack = monthly_stack.addBands(img)

    # Add static province band once per monthly image
    monthly_stack = (monthly_stack
                     .addBands(PROVINCE_RASTER)
                     .updateMask(ONTARIO_MASK))

    task = ee.batch.Export.image.toAsset(
        image       = monthly_stack,
        description = f'Monthly_{month_str}',
        assetId     = f'{EXPORT_FOLDER_MONTHLY}/Month_{month_str}',
        region      = ontario_geom,
        crs         = GRID_PROJ,
        scale       = GRID_SCALE,
        maxPixels   = 1e13
    )
    task.start()
    print(f"  Submitted: {month_str} — {len(dates)} days, ~{n_bands} bands")

print(f"\n12 monthly tasks submitted (~{12 * 1000 / 3600:.1f} EECU-hours estimated).")
print(f"Monitor: https://code.earthengine.google.com/tasks")

Already exists: projects/ru-thesis-2026/assets/Ontario_Monthly_2010-2024
Lag input collection defined.
Submitting monthly export tasks …
  Submitted: 201001 — 31 days, ~341 bands
  Submitted: 201002 — 28 days, ~308 bands
  Submitted: 201003 — 31 days, ~341 bands
  Submitted: 201004 — 30 days, ~330 bands
  Submitted: 201005 — 31 days, ~341 bands
  Submitted: 201006 — 30 days, ~330 bands
  Submitted: 201007 — 31 days, ~341 bands
  Submitted: 201008 — 31 days, ~341 bands
  Submitted: 201009 — 30 days, ~330 bands
  Submitted: 201010 — 31 days, ~341 bands
  Submitted: 201011 — 30 days, ~330 bands
  Submitted: 201012 — 31 days, ~341 bands

12 monthly tasks submitted (~3.3 EECU-hours estimated).
Monitor: https://code.earthengine.google.com/tasks
